# Explore whether we can increase the labeled data

In [62]:
"""Exploring labeled data


Structure:
    1. Imports, Variables, Functions
    2. Load Data
    3. Explore Labeled Data
"""

# 1. Imports, Variables, Functions

# imports
import sys
import os, numpy as np
import json
import h5py, scanpy as sc
sys.path.append("../..")
from src.utils import io 
import pandas as pd
from src.utils import viz as vz
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib import cm
import sys
from src.training import helpers as tr_h
import random
import seaborn as sns
import importlib
import anndata as ad
from src.utils import utils as ut
importlib.reload(tr_h)

# variables


# functions


# 2. Load Data
# load data info
df_info = ut.load_dsa_info()

# load do graph and mappings
do_g = ut.load_do_graph()
uml_2_doid = ut.get_umls_2_doid_mapping(do_g)

# 3. Explore Labeled Data
# add nº of disease samples
df_info['n_disease'] = df_info['control_case_sample_count'].map(
    lambda x: int(x.split("|")[1]) if isinstance(x, str) else None
)

# filter by species
df_info_human = df_info.query("organism == 'Homo sapiens'")
print(f"Filter {df_info_human.shape[0]} / {df_info.shape[0]} rows for Homo sapiens")
print(f"Filter {df_info_human['n_disease'].sum()} / {df_info['n_disease'].sum()} rows for Homo sapiens")


# filter by library strategy
df_info_human = df_info_human.query("library_strategy in ['Microarray', 'RNA-Seq']")
print(f"Filter {df_info_human.shape[0]} / {df_info.shape[0]} rows for Homo sapiens & microarray - RNA-Seq")
print(f"Filter {df_info_human['n_disease'].sum()} / {df_info['n_disease'].sum()} rows for Homo sapiens & microarray - RNA-Seq")

# filter by label
df_info_label = df_info_human.query(f"diseaseid in {list(uml_2_doid.keys())}")
print(f"Filter {df_info_label.shape[0]} / {df_info_human.shape[0]} rows for labeled data")
print(f"Filter {df_info_label['n_disease'].sum()} / {df_info_human['n_disease'].sum()} rows for labeled data")

df_info_unlabel = df_info_human.query(f"diseaseid not in {list(uml_2_doid.keys())}")
print(f"Filter {df_info_unlabel.shape[0]} / {df_info_human.shape[0]} rows for unlabeled data")
print(f"Filter {df_info_unlabel['n_disease'].sum()} / {df_info_human['n_disease'].sum()} rows for unlabeled data")

Filter 7194 / 10306 rows for Homo sapiens
Filter 115923 / 129491 rows for Homo sapiens
Filter 7004 / 10306 rows for Homo sapiens & microarray - RNA-Seq
Filter 115141 / 129491 rows for Homo sapiens & microarray - RNA-Seq
Filter 4681 / 7004 rows for labeled data
Filter 76869 / 115141 rows for labeled data
Filter 2323 / 7004 rows for unlabeled data
Filter 38272 / 115141 rows for unlabeled data


In [144]:

# Build mesh CUI → DOID mapping
mesh_2_doid = {}
mesh_2_doids = dict()
for node_id, data in do_g.nodes(data=True):
    xrefs = data.get('xref', [])
    for ref in xrefs:
        if ref.startswith('MESH:'):
            mesh_cui = ref.split(':', 1)[1]
            mesh_2_doid[mesh_cui] = node_id

            #! check how many have +1 mappings
            if mesh_cui not in mesh_2_doids:
                mesh_2_doids[mesh_cui] = set()
            mesh_2_doids[mesh_cui].add(node_id)

print(f"Nº of mesh w/ +1 DOID mappings: {sum([1 for v in mesh_2_doids.values() if len(v) > 1])} / {len(mesh_2_doids)}")

Nº of mesh w/ +1 DOID mappings: 242 / 3659


In [37]:
import obonet
import networkx as nx

do_graph_old = obonet.read_obo("/aloy/home/ddalton/databases/DO_FILES/HumanDiseaseOntology-2023-11-30/src/ontology/doid.obo")
do_graph_old = nx.DiGraph(do_graph_old)
do_graph_old = do_graph_old.reverse()   # reverse so leafs are at the bottom  



In [38]:

# Build UMLS CUI → DOID mapping
umls_2_doid_old = {}

for node_id, data in do_graph_old.nodes(data=True):
    xrefs = data.get('xref', [])
    for ref in xrefs:

        if ref.startswith('UMLS_CUI:'):

            umls_cui = ref.split(':', 1)[1]
            if umls_cui not in umls_2_doid_old:
                umls_2_doid_old[umls_cui] = list()
            umls_2_doid_old[umls_cui].append(node_id)


In [94]:
import obonet
import pandas as pd
import xml.etree.ElementTree as ET

mesh_umls = []

tree = ET.parse("/aloy/home/ddalton/databases/MESH_FILES/desc2025.xml")
root = tree.getroot()

# for descriptor in root.findall(".//DescriptorRecord"):
#     mesh_id = descriptor.findtext("DescriptorUI")
#     for xref in descriptor.findall(".//ConceptRelationName"):
#         if xref.text and "UMLS_CUI" in xref.text:
#             umls_id = xref.text.split(":")[1]
#             print(umls_id, mesh_id)


mesh_tree_terms_dict = {}
# Iterate through the XML to find each MeSH ID
for descriptor in root.findall(".//DescriptorRecord"):
    descriptor_id = descriptor.find("./DescriptorUI")

    tree_numbers = [tn.text for tn in descriptor.findall(".//TreeNumber")]
    mesh_tree_terms_dict[descriptor_id.text] = tree_numbers

In [104]:
import pandas as pd

# --- Path to your UMLS Metathesaurus directory ---
umls_dir = "/aloy/home/ddalton/databases/UMLS/umls-2025AA-metathesaurus-full/2025AA/META/"

# --- 1️⃣  Read only the columns you need from MRCONSO.RRF ---
# Column order per UMLS spec: 
# CUI|LAT|TS|LUI|STT|SUI|ISPREF|AUI|SAUI|SCUI|SDUI|SAB|TTY|CODE|STR|SRL|SUPPRESS|CVF
usecols = [0, 11, 13]  # CUI, SAB, CODE
colnames = ["CUI", "SAB", "CODE"]

chunks = pd.read_csv(
    umls_dir + "MRCONSO.RRF",
    sep="|",
    usecols=usecols,
    names=colnames,
    dtype=str,
    chunksize=1_000_000,
    low_memory=False,
)

# --- 2️⃣  Filter out only DO and MeSH concepts ---
dfs = []
for chunk in chunks:
    dfs.append(chunk[chunk["SAB"].isin(["MSH"])])
df = pd.concat(dfs, ignore_index=True)

# --- 3️⃣  Split into DO and MeSH tables ---
df_mesh = df[df["SAB"] == "MSH"][["CUI", "CODE"]].rename(columns={"CODE": "MESH_ID"})
df_mesh

,CUI,MESH_ID
0,C0000005,D012711
1,C0000005,D012711
2,C0000039,D015060
3,C0000039,D015060
4,C0000039,D015060
...,...,...
1030718,C5979868,D002311
1030719,C5979868,D002311
1030720,C5979868,D002311
1030721,C5979868,D002311


In [116]:
umls_2_mesh = dict(zip(df_mesh['CUI'], df_mesh['MESH_ID']))

umls_2_doid_cross = {u:mesh_2_doid.get(m) for u, m in umls_2_mesh.items() if m in mesh_2_doid}

In [ ]:
df_info_unlabel.query(f"diseaseid not in {list(umls_2_doid_cross.keys())}")["diseaseid"].nunique()

434

In [154]:
df_info_unlabel_no_cross = df_info_unlabel.query(f"diseaseid not in {list(umls_2_doid_cross.keys())}")

# save
df_info_unlabel_no_cross.to_csv(
    "tmp_outputs/df_unlabeled_no_mesh_doid_cross.csv", index=False
)

In [160]:
df_info_unlabel_no_cross["diseaseid"].nunique()

434